In [2]:
"""
MODELO — eNFN (Evolving Neo-Fuzzy Neuron)

Adaptação do algoritmo eNFN (MATLAB/Python)

O eNFN é um sistema neuro-fuzzy evolutivo que:
  - Usa funções de pertinência (FP) triangulares
  - Cria novas FPs automaticamente quando o erro local é alto
  - Remove FPs inativas (não ativadas por 'window' amostras)
  - Atualiza pesos online (amostra por amostra)
  - É originalmente um modelo de regressão; adaptamos para classificação
    aplicando threshold 0.5 na saída

Parâmetros (baseados no código MATLAB original):
  - fp = 2        (nº inicial de FPs por variável)
  - maxfp = 10    (nº máximo de FPs por variável)
  - window = 100  (janela para remoção de FPs inativas)
  - alfa0 = 1     (constante do learning rate)
  - beta = 0.01   (taxa de atualização dos centros e estatísticas)

WFA: d1=250, d2=5, modelo NOVO a cada rodada (reset completo).

Uso:
    python 04_model_eNFN.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

TRAIN_SIZE = 250
TEST_SIZE = 5

# Parâmetros eNFN (do código MATLAB original)
FP_INIT = 2       # nº inicial de funções de pertinência por variável
MAX_FP = 10       # nº máximo de FPs por variável
WINDOW = 100      # janela para exclusão de FPs inativas
ALFA0 = 1         # constante do learning rate
BETA = 0.01       # taxa de atualização
# ========================================================


class CentroStats:
    """Estatísticas de cada centro de função de pertinência."""
    __slots__ = ['n', 'mu', 'sigma', 'active']

    def __init__(self):
        self.n = 0
        self.mu = 0.0
        self.sigma = 0.01
        self.active = 0


class eNFN:
    """
    Evolving Neo-Fuzzy Neuron.
    Adaptado do código MATLAB eNFN_007_2013_01_14.m e eNFN.py.
    """

    def __init__(self, n_features, fp=FP_INIT, maxfp=MAX_FP,
                 window=WINDOW, alfa0=ALFA0, beta=BETA):
        self.n = n_features
        self.fp = fp
        self.maxfp = maxfp
        self.window = window
        self.alfa0 = alfa0
        self.beta = beta

    def fit(self, X, y):
        """
        Treina o eNFN com dados de treino (online, amostra por amostra).
        X: array (npt, n_features)
        y: array (npt,) com valores contínuos ou binários
        """
        npt, n = X.shape
        self.n = n

        # Inicialização
        xmin = X.min(axis=0)
        xmax = X.max(axis=0)
        delta = np.where(xmax > xmin, (xmax - xmin) / (self.fp - 1), 1.0)

        # Número de FPs por variável
        m = [self.fp] * n

        # Pesos q: lista de listas (dinâmica, pois nº de FPs muda)
        q = [[0.0] * self.fp for _ in range(n)]

        # Centros das FPs
        c = [[xmin[i] + j * delta[i] for j in range(self.fp)] for i in range(n)]

        # Estatísticas dos centros
        centros = [[CentroStats() for _ in range(self.fp)] for i in range(n)]

        # Estatísticas do modelo global
        modelo_mu = 0.0
        modelo_sigma = 0.003

        # Variáveis temporárias
        k1 = [0] * n
        k2 = [0] * n
        mik1 = [0.0] * n
        mik2 = [0.0] * n
        cma = [0] * n

        # --- Loop de treino (online) ---
        for k in range(npt):
            ysaida = 0.0
            alfa_sum = 0.0

            # == FUZZIFICAÇÃO ==
            for i in range(n):
                if X[k, i] <= xmin[i]:
                    mik1[i] = 1.0
                    mik2[i] = 0.0
                    k1[i] = 0
                    k2[i] = 1
                    cma[i] = 0
                elif X[k, i] >= xmax[i]:
                    mik1[i] = 0.0
                    mik2[i] = 1.0
                    k1[i] = m[i] - 2
                    k2[i] = m[i] - 1
                    cma[i] = m[i] - 1
                else:
                    for j in range(m[i] - 1):
                        if c[i][j] <= X[k, i] <= c[i][j + 1]:
                            k1[i] = j
                            ponto_medio = c[i][j] + (c[i][j + 1] - c[i][j]) / 2
                            cma[i] = j if X[k, i] <= ponto_medio else j + 1

                            # Atualiza centros (beta learning)
                            if j > 0:
                                c[i][j] += self.beta * (X[k, i] - c[i][j])
                            if j < m[i] - 2:
                                c[i][j + 1] += self.beta * (X[k, i] - c[i][j + 1])
                            break

                    a = c[i][k1[i]]
                    b = c[i][k1[i] + 1]
                    denom = a - b
                    mik1[i] = (X[k, i] - b) / denom if abs(denom) > 1e-12 else 0.5
                    k2[i] = k1[i] + 1
                    mik2[i] = 1.0 - mik1[i]

                ysaida += mik1[i] * q[i][k1[i]] + mik2[i] * q[i][k2[i]]
                alfa_sum += mik1[i] ** 2 + mik2[i] ** 2

            # == ATUALIZAÇÃO DOS PESOS ==
            dedys = ysaida - y[k]
            modelo_mu -= self.beta * (modelo_mu - dedys)
            modelo_sigma = (1 - self.beta) * (modelo_sigma + self.beta * (modelo_mu - dedys) ** 2)

            alfa = self.alfa0 / (alfa_sum + 1e-12)

            for i in range(n):
                q[i][k1[i]] -= alfa * dedys * mik1[i]
                q[i][k2[i]] -= alfa * dedys * mik2[i]

            # == CRIAÇÃO DE FPs ==
            mmu = modelo_mu
            msigma = modelo_sigma

            for i in range(n):
                ci = cma[i]
                cmu = centros[i][ci].mu
                csigma = centros[i][ci].sigma
                cn = centros[i][ci].n

                cmu -= self.beta * (cmu - dedys)
                csigma = (1 - self.beta) * (csigma + self.beta * (cmu - dedys) ** 2)
                cn += 1

                centros[i][ci].mu = cmu
                centros[i][ci].sigma = csigma
                centros[i][ci].n = cn

                if cn > 5 and cmu > (mmu + msigma):
                    espaco = (xmax[i] - xmin[i]) / self.maxfp if xmax[i] > xmin[i] else 1.0

                    if ci != 0 and ci < m[i] - 1:
                        dist = (c[i][ci + 1] - c[i][ci - 1]) / 3
                        if dist > espaco and m[i] < self.maxfp:
                            nc1 = c[i][ci - 1] + dist
                            nc2 = c[i][ci - 1] + 2 * dist
                            c[i][ci] = nc1
                            centros[i][ci] = CentroStats()
                            centros[i][ci].active = k + 1
                            m[i] += 1
                            c[i].insert(ci + 1, nc2)
                            q[i].insert(ci + 1, q[i][ci])
                            centros[i].insert(ci + 1, CentroStats())
                            centros[i][ci + 1].active = k + 1

                    elif ci == 0:
                        dist = (c[i][1] - c[i][0]) / 2
                        if dist > espaco and m[i] < self.maxfp:
                            nc = c[i][0] + dist
                            m[i] += 1
                            c[i].insert(1, nc)
                            q[i].insert(1, q[i][0])
                            centros[i].insert(1, CentroStats())
                            centros[i][1].active = k + 1

                    elif ci == m[i] - 1:
                        dist = (c[i][ci] - c[i][ci - 1]) / 2
                        if dist > espaco and m[i] < self.maxfp:
                            nc = c[i][ci] - dist
                            m[i] += 1
                            c[i].insert(ci, nc)
                            q[i].insert(ci, q[i][ci])
                            centros[i].insert(ci, CentroStats())
                            centros[i][ci].active = k + 1

                # == ATUALIZAÇÃO DE ATIVIDADE ==
                centros[i][k1[i]].active = k + 1
                centros[i][k2[i]].active = k + 1

                # == EXCLUSÃO DE FPs INATIVAS ==
                if m[i] > 2:
                    # Encontra centro menos ativado
                    min_active = centros[i][0].active
                    idx_min = 0
                    for j in range(1, m[i]):
                        if centros[i][j].active < min_active:
                            min_active = centros[i][j].active
                            idx_min = j

                    if (k + 1) - min_active > self.window:
                        c[i].pop(idx_min)
                        q[i].pop(idx_min)
                        centros[i].pop(idx_min)
                        m[i] -= 1

        # Salvar estado treinado para predição
        self._m = m
        self._q = q
        self._c = c
        self._xmin = xmin
        self._xmax = xmax

    def predict(self, X):
        """
        Prediz com o modelo treinado (sem atualização de pesos).
        X: array (npt, n_features)
        Retorna: array de saídas contínuas
        """
        npt = X.shape[0]
        n = self.n
        m = self._m
        q = self._q
        c = self._c
        xmin = self._xmin
        xmax = self._xmax

        outputs = np.zeros(npt)

        for k in range(npt):
            ysaida = 0.0

            for i in range(n):
                if X[k, i] <= xmin[i]:
                    mik1_val = 1.0
                    mik2_val = 0.0
                    k1_val = 0
                    k2_val = 1
                elif X[k, i] >= xmax[i]:
                    mik1_val = 0.0
                    mik2_val = 1.0
                    k1_val = m[i] - 2
                    k2_val = m[i] - 1
                else:
                    k1_val = 0
                    for j in range(m[i] - 1):
                        if c[i][j] <= X[k, i] <= c[i][j + 1]:
                            k1_val = j
                            break

                    a = c[i][k1_val]
                    b = c[i][k1_val + 1]
                    denom = a - b
                    mik1_val = (X[k, i] - b) / denom if abs(denom) > 1e-12 else 0.5
                    k2_val = k1_val + 1
                    mik2_val = 1.0 - mik1_val

                # Limitar índices
                k1_val = max(0, min(k1_val, len(q[i]) - 1))
                k2_val = max(0, min(k2_val, len(q[i]) - 1))

                ysaida += mik1_val * q[i][k1_val] + mik2_val * q[i][k2_val]

            outputs[k] = ysaida

        return outputs

    def predict_binary(self, X, threshold=0.5):
        """Prediz classes binárias (0/1) via threshold."""
        raw = self.predict(X)
        return (raw >= threshold).astype(int)


def read_codes(path: Path) -> list:
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    return df["Code"].str.strip().str.upper().tolist()


def run_wfa_enfn(code: str, base_dir: Path) -> dict:
    """Executa Walk-Forward Analysis com eNFN para um ticker."""
    infile = base_dir / f"{code}_DatasetNew.csv"
    outfile = base_dir / f"{code}_TradeSignal_eNFN.csv"

    if outfile.exists():
        return {"Code": code, "status": "skipped", "signals": 0}

    if not infile.exists():
        return {"Code": code, "status": "no_DatasetNew", "signals": 0}

    try:
        df = pd.read_csv(infile, encoding="utf-8-sig")
    except Exception as e:
        return {"Code": code, "status": f"read_error: {e}", "signals": 0}

    if df.shape[1] < 3:
        return {"Code": code, "status": "too_few_columns", "signals": 0}

    date_col = df.columns[0]
    label_col = df.columns[-1]
    feature_cols = df.columns[1:-1].tolist()
    n_features = len(feature_cols)

    # --- Alinhamento WFA (idêntico ao R e aos outros modelos) ---
    M = len(df)
    if M < TRAIN_SIZE + TEST_SIZE:
        return {"Code": code, "status": f"too_few_rows ({M})", "signals": 0}

    Q = (M - TRAIN_SIZE) // TEST_SIZE
    H = (M - TRAIN_SIZE) - TEST_SIZE * Q
    df = df.iloc[H:].reset_index(drop=True)

    dates = df[date_col].values
    X = df[feature_cols].values.astype(float)
    y = df[label_col].values.astype(float)

    predict_signal = []
    predict_dates = []

    # --- Loop WFA ---
    for i in range(Q):
        train_start = i * TEST_SIZE
        train_end = train_start + TRAIN_SIZE
        test_start = train_end
        test_end = test_start + TEST_SIZE

        X_train = X[train_start:train_end]
        y_train = y[train_start:train_end]
        X_test = X[test_start:test_end]
        test_dates = dates[test_start:test_end]

        if len(np.unique(y_train.astype(int))) < 2:
            preds = [int(y_train[0])] * len(X_test)
        else:
            try:
                # Modelo NOVO a cada rodada (reset completo)
                model = eNFN(n_features)
                model.fit(X_train, y_train)
                preds = model.predict_binary(X_test, threshold=0.5).tolist()
            except Exception:
                preds = [0] * len(X_test)

        predict_signal.extend(preds)
        predict_dates.extend(test_dates)

    # --- Salvar ---
    if predict_signal:
        df_out = pd.DataFrame({"Date": predict_dates, "pre_signal": predict_signal})
        df_out.to_csv(outfile, index=False, encoding="utf-8-sig")

    return {"Code": code, "status": "ok", "signals": len(predict_signal)}


def main():
    codes = read_codes(SEC_NAMES)
    print(f"Modelo: eNFN (Evolving Neo-Fuzzy Neuron)")
    print(f"Parâmetros: fp={FP_INIT}, maxfp={MAX_FP}, window={WINDOW}, beta={BETA}")
    print(f"WFA: d1={TRAIN_SIZE}, d2={TEST_SIZE}")
    print(f"Tickers: {len(codes)}\n")

    report = []
    for code in tqdm(codes, desc="eNFN Walk-Forward"):
        result = run_wfa_enfn(code, BASE_DIR)
        report.append(result)

    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "model_eNFN_report.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    avg_signals = report_df.loc[report_df["status"] == "ok", "signals"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} processados, {n_skip} já existiam.")
    print(f"Média de sinais por ação: {avg_signals:.0f}")
    print(f"Relatório: model_eNFN_report.csv")


if __name__ == "__main__":
    main()

Modelo: eNFN (Evolving Neo-Fuzzy Neuron)
Parâmetros: fp=2, maxfp=10, window=100, beta=0.01
WFA: d1=250, d2=5
Tickers: 246



eNFN Walk-Forward: 100%|██████████| 246/246 [24:48<00:00,  6.05s/it]


Concluído: 246 processados, 0 já existiam.
Média de sinais por ação: 550
Relatório: model_eNFN_report.csv
